# 04. Ajuste de hiperparámetros y modelo final


Ajustamos los dos mejores modelos del notebook 03 con búsqueda en rejilla / aleatoria y CV de 5 folds, y probamos una mezcla de ambos.

In [ ]:
import sys
from pathlib import Path

# Permite importar el paquete src/ desde la carpeta notebooks/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from src.config import DATA_RAW, TARGET, CATEGORICAL, NUMERIC
sns.set_theme(style="whitegrid")
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from src.preprocessing import add_features, split_xy
from src.models import get_cv, evaluar_cv, rmse, ridge_log, lgbm_log, MezclaRegressor

In [ ]:
train = pd.read_csv(DATA_RAW / "train.csv")
X, y = split_xy(train)
X = add_features(X)

## 1. Ridge (log price): parámetro `alpha`

`alpha` controla la regularización: más alto = coeficientes más pequeños.

In [ ]:
grid = GridSearchCV(ridge_log(), {"regressor__model__alpha": [0.1, 0.3, 1, 3, 10, 30]},
                    cv=get_cv(), scoring="neg_root_mean_squared_error")
grid.fit(X, y)
print("Mejor alpha:", grid.best_params_, " RMSE CV:", round(-grid.best_score_, 3))
pd.DataFrame(grid.cv_results_)[["param_regressor__model__alpha", "mean_test_score"]].assign(rmse=lambda d: -d.mean_test_score)

## 2. LightGBM (log price): búsqueda aleatoria

Probamos 15 combinaciones al azar (tarda unos minutos).

In [ ]:
espacio = {
    "regressor__model__n_estimators": [400, 800, 1200],
    "regressor__model__learning_rate": [0.01, 0.02, 0.04],
    "regressor__model__num_leaves": [3, 5, 7, 15],
    "regressor__model__min_child_samples": [20, 40, 80],
    "regressor__model__colsample_bytree": [0.5, 0.7, 0.9],
    "regressor__model__reg_lambda": [0, 1, 5],
}
busqueda = RandomizedSearchCV(lgbm_log(), espacio, n_iter=15, cv=get_cv(),
                              scoring="neg_root_mean_squared_error", random_state=42)
busqueda.fit(X, y)
print("Mejores parámetros:", busqueda.best_params_)
print("RMSE CV:", round(-busqueda.best_score_, 3))

## 3. Mezcla de los dos modelos

Combinamos `peso * Ridge + (1 - peso) * LightGBM` y elegimos el peso con las predicciones *out-of-fold*.

In [ ]:
_, oof_ridge = evaluar_cv(grid.best_estimator_, X, y)
_, oof_lgbm = evaluar_cv(busqueda.best_estimator_, X, y)
pesos = np.round(np.arange(0, 1.01, 0.1), 1)
curva = [rmse(y, w * oof_ridge + (1 - w) * oof_lgbm) for w in pesos]
plt.plot(pesos, curva, marker="o"); plt.xlabel("peso de Ridge"); plt.ylabel("RMSE OOF"); plt.show()
print("Mejor peso:", pesos[int(np.argmin(curva))])

## 4. Modelo final

Los valores elegidos se copian a `src/modelo.py` (`MEJOR_ALPHA_RIDGE`, `MEJORES_PARAMS_LGBM`, `PESO_RIDGE`). Comprobamos el RMSE del modelo final:

In [ ]:
from src.modelo import build_final_model
por_fold, _ = evaluar_cv(build_final_model(), X, y)
print("RMSE por fold:", por_fold.round(2))
print(f"RMSE CV modelo final: {por_fold.mean():.3f} ± {por_fold.std():.3f}")

Para generar la submission, desde la raíz del repo:

```
python -m src.train NombreAlumno
```

Crea `submissions/submission_NombreAlumno.csv` con columnas `id,prediction` (separador `;`, igual que `submission_ejemplo.csv`).